# 06｜Swin Block 完整结构

前面已经分别学过 W-MSA、SW-MSA、mask 和相对位置偏置。现在把 Attention 外面的组件补齐，组成一个完整 Swin Block。

这一课会先解释每个零件，再组装，不直接从公式开始。

## 1. Block 是什么

Block 可以理解成网络中重复使用的一个处理单元。

一个 Swin Block 接收一张 token 特征图，更新每个 token 的信息，再输出相同 shape 的特征图。

因为输入输出 shape 相同，多个 Blocks 可以像积木一样连续堆叠。

## 2. LayerNorm 是什么

LayerNorm 会对单个 token 的特征维度进行归一化，使特征数值保持在比较稳定的范围。

它不会让 tokens 互相交流，也不会改变 shape。它主要为后面的 Attention 或 MLP 准备更稳定的输入。

Swin 在 Attention 和 MLP 以前使用 LayerNorm，这种顺序叫 Pre-Norm。名字不重要，先记住“先归一化，再计算”。

## 3. 残差连接是什么

残差连接会把子层处理前的输入，直接加回子层输出。

可以把它理解成两条路：

- 主路：经过 Attention 或 MLP，学习新的变化；
- 快路：原始信息直接通过；
- 最后把两条路相加。

这样模型不用每一层都重新创造全部表示，只需要在原表示上学习应该增加或修改什么。

## 4. MLP 是什么

Attention 负责让不同 tokens 交换信息。MLP 负责在每个 token 内部加工特征。

Swin 中的 MLP 通常先把通道维度扩大到约 4 倍，经过 GELU 激活，再压回原通道数。

同一个 MLP 会独立应用到每个 token，参数共享。它不会改变 token 数量和空间位置。

## 5. 一个完整 Swin Block 的顺序

一个 Block 按下面顺序处理：

1. LayerNorm：整理每个 token 的特征数值。
2. W-MSA 或 SW-MSA：让 tokens 在当前窗口分组中交换信息。
3. 第一次残差连接：把 Attention 以前的输入加回来。
4. LayerNorm：再次整理特征数值。
5. MLP：独立加工每个 token 的通道特征。
6. 第二次残差连接：把 MLP 以前的输入加回来。

Attention 负责 token 之间，MLP 负责 token 内部；两次残差连接分别保护两部分的输入信息。

![完整 Swin Block](images/07_swin_block.svg)

## 6. 两种 Swin Block

Swin 中有两种交替出现的 Block：

| 位置 | Attention 子层 | 作用 |
|---|---|---|
| 第一个 Block | W-MSA | 固定窗口内部交流 |
| 第二个 Block | SW-MSA | 移动窗口后跨原边界交流 |

除了 Attention 的窗口分组方式不同，它们的 LayerNorm、残差连接和 MLP 结构相同。

## 7. 一对 Blocks 怎样工作

第一个 Block 先用 W-MSA 汇总每个规则窗口内部的信息，再用 MLP 加工每个 token。

第二个 Block 接收第一个 Block 的结果，用 SW-MSA 让相邻窗口交换已经汇总过的信息，然后再次用 MLP 加工。

所以一对 Blocks 完成：局部窗口内部融合 → 跨窗口融合。

继续堆叠更多 Block 对，信息就会逐渐传播到更远位置。

## 8. Block 中什么会改变，什么不会改变

| 项目 | 是否改变 |
|---|---|
| token 的特征内容 | 会 |
| token 数量 | 不会 |
| 高和宽 | 不会 |
| 通道数 | 不会 |

这也是残差连接能够直接相加的原因：两条路径的 shape 必须一致。

只有 Stage 之间的 Patch Merging 才会改变高、宽和通道数。

## 9. DropPath 简单了解

训练较深网络时，Swin 还常使用 DropPath。

它会在训练过程中随机跳过某些残差分支，减少模型过度依赖固定路径，起到正则化作用。

推理时不会随机跳过。理解 Swin 主体结构时，可以先把它看成残差分支上的训练辅助方法，不影响 W-MSA 与 SW-MSA 的核心逻辑。

## 10. 本节小结

1. Swin Block 是可重复堆叠的处理单元，输入输出 shape 相同。
2. LayerNorm 稳定特征，Attention 负责 token 间交流，MLP 负责 token 内加工。
3. Attention 和 MLP 外面各有一次残差连接。
4. 相邻 Blocks 分别使用 W-MSA 与 SW-MSA。
5. Block 不改变空间尺寸；Patch Merging 才负责下采样。

## 11. 自测问题

1. Block 为什么可以连续堆叠？
2. LayerNorm 的主要作用是什么？
3. Attention 与 MLP 的分工有什么不同？
4. 一个 Block 为什么需要两次残差连接？
5. W-MSA Block 与 SW-MSA Block 哪部分不同？
6. Swin Block 会不会改变高和宽？
7. DropPath 在训练中起什么作用？

### 自测参考答案

1. 输入输出 shape 相同。
2. 稳定单个 token 的特征数值，为后续计算准备输入。
3. Attention 让 tokens 交换信息，MLP 独立加工每个 token 的特征。
4. 分别让 Attention 和 MLP 在保留原信息的基础上学习变化。
5. Attention 使用的窗口分组方式不同。
6. 不会。
7. 随机跳过部分残差分支，起正则化作用。